# 04 PEFT Fine-Tuning: FLAN-T5 + LoRA on Colab

This notebook fine-tunes `google/flan-t5-base` on the ScholarSynth instruction dataset using LoRA. It is designed for Google Colab because local fine-tuning and RAG evaluation can be slow on CPU.

Expected input files:

- `data/finetune_train.jsonl`
- `data/finetune_val.jsonl`
- `data/finetune_test.jsonl`

Output adapter path:

- `models/flan_t5_lora/`


## 1. Check GPU

In Colab, use `Runtime > Change runtime type > T4 GPU` before running the notebook.

In [ ]:
!nvidia-smi


## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate rouge-score nltk sentencepiece sacrebleu


## 3. Mount Google Drive

Recommended Drive layout:

```text
/content/drive/MyDrive/ScholarSynth-AI/
  data/
    finetune_train.jsonl
    finetune_val.jsonl
    finetune_test.jsonl
  models/
```


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ScholarSynth-AI'
DATA_DIR = f'{PROJECT_DIR}/data'
OUTPUT_DIR = f'{PROJECT_DIR}/models/flan_t5_lora'

TRAIN_PATH = f'{DATA_DIR}/finetune_train.jsonl'
VAL_PATH = f'{DATA_DIR}/finetune_val.jsonl'
TEST_PATH = f'{DATA_DIR}/finetune_test.jsonl'

PROJECT_DIR, TRAIN_PATH, VAL_PATH, TEST_PATH, OUTPUT_DIR


## 4. Load Dataset

In [ ]:
import json
from pathlib import Path

from datasets import Dataset, DatasetDict


def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not Path(path).exists():
        raise FileNotFoundError(f'Missing file: {path}')

raw_datasets = DatasetDict({
    'train': Dataset.from_list(load_jsonl(TRAIN_PATH)),
    'validation': Dataset.from_list(load_jsonl(VAL_PATH)),
    'test': Dataset.from_list(load_jsonl(TEST_PATH)),
})

raw_datasets


In [ ]:
raw_datasets['train'][0]


## 5. Build Training Prompts

The same prompt style should later be used during evaluation and inference.

In [ ]:
def build_source_text(example):
    return (
        'You are ScholarSynth AI, an academic research assistant. '
        'Write a concise answer grounded only in the given input.\n\n'
        f"Task: {example['task']}\n"
        f"Topic: {example['topic']}\n"
        f"Instruction: {example['instruction']}\n"
        f"Input:\n{example['input']}\n\n"
        'Answer:'
    )


def add_prompt_fields(example):
    return {
        'source_text': build_source_text(example),
        'target_text': example['output'],
    }


prompt_datasets = raw_datasets.map(add_prompt_fields)
prompt_datasets['train'][0]['source_text'][:800]


## 6. Tokenize

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = 'google/flan-t5-base'
MAX_INPUT_LENGTH = 768
MAX_TARGET_LENGTH = 192

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['source_text'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs


tokenized_datasets = prompt_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=prompt_datasets['train'].column_names,
)

tokenized_datasets


## 7. Load FLAN-T5 with LoRA

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q', 'v'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)


## 8. Train

The settings below are conservative for a Colab T4. Increase `num_train_epochs` to 3-5 if time allows.

In [ ]:
import os
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=f'{OUTPUT_DIR}/trainer_checkpoints',
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


## 9. Save LoRA Adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'Saved LoRA adapter and tokenizer to: {OUTPUT_DIR}')
!ls -lh "$OUTPUT_DIR"


## 10. Quick Generation Test

In [ ]:
from peft import PeftModel

inference_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
inference_model = PeftModel.from_pretrained(inference_base, OUTPUT_DIR)
inference_model.eval()

sample = prompt_datasets['test'][0]
inputs = tokenizer(sample['source_text'], return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH)

with torch.inference_mode():
    output_ids = inference_model.generate(
        **inputs,
        max_new_tokens=160,
        num_beams=4,
        do_sample=False,
    )

prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print('PROMPT:\n', sample['source_text'][:1000])
print('\nREFERENCE:\n', sample['target_text'])
print('\nPREDICTION:\n', prediction)


## 11. Evaluate on Test Split

This lightweight evaluation computes ROUGE and BLEU. BERTScore is slower, so run it separately if needed.

In [ ]:
import evaluate
import numpy as np
import pandas as pd

rouge = evaluate.load('rouge')
bleu = evaluate.load('sacrebleu')


def generate_prediction(source_text):
    inputs = tokenizer(
        source_text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )
    if torch.cuda.is_available():
        inputs = {key: value.cuda() for key, value in inputs.items()}
        inference_model.cuda()
    with torch.inference_mode():
        output_ids = inference_model.generate(
            **inputs,
            max_new_tokens=160,
            num_beams=4,
            do_sample=False,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


test_examples = prompt_datasets['test'].select(range(min(40, len(prompt_datasets['test']))))
predictions = [generate_prediction(item['source_text']) for item in test_examples]
references = [item['target_text'] for item in test_examples]

rouge_scores = rouge.compute(predictions=predictions, references=references)
bleu_score = bleu.compute(predictions=predictions, references=[[ref] for ref in references])

metrics = {
    'model': 'fine_tuned_lora',
    'eval_examples': len(test_examples),
    'bleu': bleu_score['score'],
    'rouge1': rouge_scores['rouge1'],
    'rouge2': rouge_scores['rouge2'],
    'rougeL': rouge_scores['rougeL'],
}

metrics_df = pd.DataFrame([metrics])
metrics_path = f'{PROJECT_DIR}/outputs/lora_test_metrics.csv'
Path(f'{PROJECT_DIR}/outputs').mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(metrics_path, index=False)

generations_df = pd.DataFrame({
    'reference': references,
    'prediction': predictions,
})
generations_path = f'{PROJECT_DIR}/outputs/lora_test_generations.csv'
generations_df.to_csv(generations_path, index=False)

display(metrics_df)
print(f'Saved metrics to: {metrics_path}')
print(f'Saved generations to: {generations_path}')


## 12. Use the Adapter Locally in VS Code

After training, copy the saved adapter folder from Drive back into the project:

```text
models/flan_t5_lora/
```

Then load it with:

```python
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

base = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
model = PeftModel.from_pretrained(base, 'models/flan_t5_lora')
tokenizer = AutoTokenizer.from_pretrained('models/flan_t5_lora')
```
